In [1]:
import sys, time, csv
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "requirements.txt").exists())
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
from src.models.embeddings import encode_news

RAW = ROOT / "data" / "raw"
SALIDA = ROOT / "data" / "processed" / "large"
SALIDA.mkdir(parents=True, exist_ok=True)

MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
COLS = ["news_id", "category", "subcategory", "title", "abstract",
        "url", "title_entities", "abstract_entities"]

for s in ("train", "dev"):
    for f in ("news.tsv", "behaviors.tsv"):
        p = RAW / f"MINDlarge_{s}" / f
        print(f"{str(p.relative_to(ROOT)):45s} {'OK' if p.exists() else 'NO ENCONTRADO'}"
              + (f"  {p.stat().st_size / 1e6:8.1f} MB" if p.exists() else ""))

data\raw\MINDlarge_train\news.tsv             OK      84.9 MB
data\raw\MINDlarge_train\behaviors.tsv        OK    1373.8 MB
data\raw\MINDlarge_dev\news.tsv               OK      59.1 MB
data\raw\MINDlarge_dev\behaviors.tsv          OK     230.7 MB


In [2]:
news = pd.concat([
    pd.read_csv(RAW / f"MINDlarge_{s}" / "news.tsv", sep="\t", header=None, names=COLS,
                quoting=csv.QUOTE_NONE, usecols=["news_id", "title", "abstract"])
    for s in ("train", "dev")
]).drop_duplicates("news_id").reset_index(drop=True)

news["text"] = (news["title"].fillna("") + " " + news["abstract"].fillna("")).str.strip()

print(f"Noticias únicas: {len(news):,}")
print(f"Abstracts vacíos: {(news['abstract'].fillna('').str.strip() == '').mean():.1%}")
news.head(3)

Noticias únicas: 104,151
Abstracts vacíos: 5.4%


,news_id,title,abstract,text
0,N88753,"The Brands Queen Elizabeth, Prince Charles, an...","Shop the notebooks, jackets, and more that the...","The Brands Queen Elizabeth, Prince Charles, an..."
1,N45436,Walmart Slashes Prices on Last-Generation iPads,Apple's new iPad releases bring big deals on l...,Walmart Slashes Prices on Last-Generation iPad...
2,N23144,50 Worst Habits For Belly Fat,These seemingly harmless habits are holding yo...,50 Worst Habits For Belly Fat These seemingly ...


In [3]:
t0 = time.time()
emb = encode_news(news, MODEL_NAME)
print(f"Embeddings: {emb.shape} en {time.time() - t0:.0f} s ({(time.time() - t0) / 60:.1f} min)")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/814 [00:00<?, ?it/s]

Embeddings: (104151, 384) en 1840 s (30.7 min)


In [4]:
destino = SALIDA / "news_emb_minilm.parquet"
pd.DataFrame({"news_id": news["news_id"], "emb": list(emb.astype(np.float32))}) \
  .to_parquet(destino, index=False)

print(f"Guardado: {destino}  ({destino.stat().st_size / 1e6:.0f} MB)")

Guardado: c:\DevProjects\MMDS\RecSysMIND\data\processed\large\news_emb_minilm.parquet  (161 MB)


In [5]:
chk = pd.read_parquet(destino)
v = np.stack(chk["emb"].head(1000).to_numpy())
print(f"Filas: {len(chk):,} | dimensión: {v.shape[1]} | "
      f"norma media: {np.linalg.norm(v, axis=1).mean():.4f} (debe ser ≈ 1)")

Filas: 104,151 | dimensión: 384 | norma media: 1.0000 (debe ser ≈ 1)
